%md
# Eligibility Reconciliation - PBM vs TPA

The PBM and each TPA both maintain their own copy of member eligibility,
loaded from the same 834 enrollment file sent by the plan sponsor. In theory
they should agree. In practice they rarely do, because the 834 gets processed
at different times, parsed with different logic, and sometimes one side
receives an update that the other hasn't ingested yet.

When eligibility is out of sync, claims get adjudicated against the wrong
status. A member who is terminated on the TPA side but still active on the
PBM side can fill prescriptions that should have been rejected. A member
missing from the TPA feed won't have their medical accumulators reported,
which means the PBM is adjudicating without knowing what the member has
already spent on the medical side.

This notebook compares `pbm_eligibility` against `tpa_eligibility_feed`
and flags every discrepancy that needs analyst attention.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261017_191803


In [0]:
from pyspark.sql import functions as F

spark.sql("USE pbm_sim")

NB = "NB_02"

# Pre-flight - make sure both eligibility tables exist before we try to compare them.
# If the TPA feed didn't land (SFTP failure, upstream job skipped), there's nothing
# to reconcile and we'd rather know that now than get an AnalysisException later.
for tbl in ["pbm_sim.pbm_eligibility", "pbm_sim.tpa_eligibility_feed"]:
    validate_table_exists(tbl, NB)

df_pbm = spark.table("pbm_sim.pbm_eligibility")
df_tpa = spark.table("pbm_sim.tpa_eligibility_feed")

log_entry(NB, "load_source_tables", "SUCCESS",
          detail=f"PBM eligibility: {df_pbm.count()} records, "
                 f"TPA eligibility: {df_tpa.count()} records")

print(f"PBM eligibility records: {df_pbm.count()}")
print(f"TPA eligibility records: {df_tpa.count()}")
print(f"Difference: {df_pbm.count() - df_tpa.count()}")

[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:06.392068+00:00", "notebook": "NB_02", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "PBM eligibility: 49167 records, TPA eligibility: 49042 records", "duration_sec": 0.0}
PBM eligibility records: 49167
TPA eligibility records: 49042
Difference: 125


## 1. Members Present in Only One System

A member who exists in PBM but not in TPA is a gap in medical accumulator
reporting. We won't receive their medical-side spending, so the PBM
adjudicates claims as if the member has zero medical deductible applied.
If the member has actually been spending on the medical side, the PBM
will overcharge them.

The reverse (in TPA but not PBM) is less common but equally problematic.
The TPA is reporting accumulators for someone the PBM doesn't recognize.

In [0]:
df_pbm_ids = df_pbm.select("member_id").distinct()
df_tpa_ids = df_tpa.select("member_id").distinct()

# Members in PBM but missing from TPA
df_in_pbm_only = df_pbm_ids.join(df_tpa_ids, on="member_id", how="left_anti")
in_pbm_only_count = df_in_pbm_only.count()

# Members in TPA but missing from PBM
df_in_tpa_only = df_tpa_ids.join(df_pbm_ids, on="member_id", how="left_anti")
in_tpa_only_count = df_in_tpa_only.count()

print(f"In PBM only (missing from TPA): {in_pbm_only_count}")
print(f"In TPA only (missing from PBM): {in_tpa_only_count}")

log_entry(NB, "members_pbm_only", "WARNING" if in_pbm_only_count > 0 else "SUCCESS",
          row_count=in_pbm_only_count,
          detail=f"{in_pbm_only_count} members in PBM but absent from TPA feed")

log_entry(NB, "members_tpa_only", "WARNING" if in_tpa_only_count > 0 else "SUCCESS",
          row_count=in_tpa_only_count,
          detail=f"{in_tpa_only_count} members in TPA but absent from PBM")

In PBM only (missing from TPA): 125
In TPA only (missing from PBM): 0
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:12.119386+00:00", "notebook": "NB_02", "section": "members_pbm_only", "status": "WARNING", "row_count": 125, "detail": "125 members in PBM but absent from TPA feed", "duration_sec": 0.0}
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:13.613707+00:00", "notebook": "NB_02", "section": "members_tpa_only", "status": "SUCCESS", "row_count": 0, "detail": "0 members in TPA but absent from PBM", "duration_sec": 0.0}


In [0]:
# Show the PBM-only members with their plan and enrollment details.
# In production this is the list you'd send to the TPA and ask them
# to confirm enrollment status.
if in_pbm_only_count > 0:
    print("Members in PBM eligibility but absent from TPA feed:")
    df_pbm_only_detail = (
        df_pbm
        .join(df_in_pbm_only, on="member_id")
        .select(
            "member_id", "plan_id", "first_name", "last_name",
            "effective_date", "term_date", "relationship"
        )
        .orderBy("plan_id", "member_id")
    )
    display(df_pbm_only_detail)

Members in PBM eligibility but absent from TPA feed:


member_id,plan_id,first_name,last_name,effective_date,term_date,relationship
MBR1000266,PLN001,Shirley,Diaz,2026-01-01,null,SUBSCRIBER
MBR1000302,PLN001,Henry,Lopez,2026-01-01,null,SUBSCRIBER
MBR1000336,PLN001,Zachary,Anderson,2026-01-01,null,SUBSCRIBER
MBR1000829,PLN001,Zachary,Baker,2026-01-01,null,SUBSCRIBER
MBR1000999,PLN001,Rebecca,Nelson,2026-01-01,null,SPOUSE
MBR1002032,PLN001,Helen,Cox,2026-01-01,null,DEPENDENT
MBR1002102,PLN001,Jonathan,Kelly,2026-01-01,null,DEPENDENT
MBR1002374,PLN002,Brian,Gonzalez,2026-01-01,null,DEPENDENT
MBR1002731,PLN002,Ryan,Murphy,2026-01-01,null,SPOUSE
MBR1002940,PLN002,Catherine,Taylor,2026-01-01,null,SUBSCRIBER


## 2. Termination Status Mismatches

This is one potentially dangerous mismatch. If the TPA has terminated
a member but the PBM still shows them as active, the member can continue
filling prescriptions on a plan they're no longer enrolled in. The plan
sponsor eats those costs, and unwinding the claims after the fact requires
manual reversal and member outreach.

The root cause is almost always timing. The plan sponsor sends an 834
with the termination. The TPA processes it within hours. The PBM processes
it on a different cycle, sometimes days later. In that window, the systems
disagree.

In [0]:
# Join on member_id to compare term_date across systems.
# We only look at members that exist in BOTH systems - the ones that are
# missing entirely were already flagged in check 1.
df_joined = (
    df_pbm.alias("pbm")
    .join(df_tpa.alias("tpa"), on="member_id", how="inner")
    .select(
        F.col("pbm.member_id"),
        F.col("pbm.plan_id"),
        F.col("pbm.first_name"),
        F.col("pbm.last_name"),
        F.col("pbm.term_date").alias("pbm_term_date"),
        F.col("tpa.term_date").alias("tpa_term_date"),
        F.col("pbm.effective_date").alias("pbm_effective_date"),
        F.col("tpa.effective_date").alias("tpa_effective_date"),
    )
)

# Case 1: Active in PBM (term_date is null) but terminated in TPA (term_date is not null).
# This is the critical one - member might be filling claims they shouldn't be.
df_pbm_active_tpa_termed = df_joined.filter(
    F.col("pbm_term_date").isNull() & F.col("tpa_term_date").isNotNull()
)

# Case 2: Terminated in PBM but active in TPA.
# Less urgent but still needs cleanup - TPA is reporting accumulators
# for someone the PBM considers gone.
df_pbm_termed_tpa_active = df_joined.filter(
    F.col("pbm_term_date").isNotNull() & F.col("tpa_term_date").isNull()
)

# Case 3: Both terminated but on different dates.
# Usually a timing issue in how the 834 was processed.
df_term_date_mismatch = df_joined.filter(
    F.col("pbm_term_date").isNotNull() &
    F.col("tpa_term_date").isNotNull() &
    (F.col("pbm_term_date") != F.col("tpa_term_date"))
)

pbm_active_tpa_termed_count = df_pbm_active_tpa_termed.count()
pbm_termed_tpa_active_count = df_pbm_termed_tpa_active.count()
term_date_mismatch_count = df_term_date_mismatch.count()

print(f"Active in PBM, terminated in TPA:    {pbm_active_tpa_termed_count}")
print(f"Terminated in PBM, active in TPA:    {pbm_termed_tpa_active_count}")
print(f"Both termed, different dates:        {term_date_mismatch_count}")

log_entry(NB, "term_pbm_active_tpa_termed",
          "WARNING" if pbm_active_tpa_termed_count > 0 else "SUCCESS",
          row_count=pbm_active_tpa_termed_count,
          detail=f"{pbm_active_tpa_termed_count} members active in PBM but termed in TPA")

log_entry(NB, "term_pbm_termed_tpa_active",
          "WARNING" if pbm_termed_tpa_active_count > 0 else "SUCCESS",
          row_count=pbm_termed_tpa_active_count,
          detail=f"{pbm_termed_tpa_active_count} members termed in PBM but active in TPA")

log_entry(NB, "term_date_mismatch",
          "WARNING" if term_date_mismatch_count > 0 else "SUCCESS",
          row_count=term_date_mismatch_count,
          detail=f"{term_date_mismatch_count} members with conflicting term dates")

Active in PBM, terminated in TPA:    500
Terminated in PBM, active in TPA:    0
Both termed, different dates:        0
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:24.021081+00:00", "notebook": "NB_02", "section": "term_pbm_active_tpa_termed", "status": "WARNING", "row_count": 500, "detail": "500 members active in PBM but termed in TPA", "duration_sec": 0.0}
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:25.435400+00:00", "notebook": "NB_02", "section": "term_pbm_termed_tpa_active", "status": "SUCCESS", "row_count": 0, "detail": "0 members termed in PBM but active in TPA", "duration_sec": 0.0}
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:26.849457+00:00", "notebook": "NB_02", "section": "term_date_mismatch", "status": "SUCCESS", "row_count": 0, "detail": "0 members with conflicting term dates", "duration_sec": 0.0}


In [0]:
# A critical case happens when the PBM thinks some members are still active but the TPA has already terminated.
# These members may be filling prescriptions on a plan they've left.
if pbm_active_tpa_termed_count > 0:
    print("CRITICAL: Members still active in PBM but already terminated per TPA:")
    display(
        df_pbm_active_tpa_termed
        .select(
            "member_id", "plan_id", "first_name", "last_name",
            "pbm_term_date", "tpa_term_date"
        )
        .orderBy("tpa_term_date", "plan_id")
    )

CRITICAL: Members still active in PBM but already terminated per TPA:


member_id,plan_id,first_name,last_name,pbm_term_date,tpa_term_date
MBR1007178,PLN004,Brenda,Bailey,null,2026-01-31
MBR1039740,PLN028,Carolyn,Lopez,null,2026-01-31
MBR1027227,PLN016,Ryan,Thompson,null,2026-02-01
MBR1027226,PLN016,Christine,Thompson,null,2026-02-01
MBR1003804,PLN002,Brenda,Ortiz,null,2026-02-02
MBR1009404,PLN005,Mark,Cox,null,2026-02-02
MBR1042246,PLN031,Sharon,Rogers,null,2026-02-02
MBR1031283,PLN020,Jonathan,Roberts,null,2026-02-03
MBR1033182,PLN022,Michael,Murphy,null,2026-02-03
MBR1037816,PLN026,Kenneth,Reyes,null,2026-02-03


### Post-Termination Claims

For each of those members where the PBM still thinks they're active,
we need to check whether any pharmacy claims were adjudicated after
the TPA's termination date. If so, those claims were processed for a
member who should not have been eligible, and the plan sponsor will
want them reversed or recovered.

In [0]:
# Check if any claims were filled after the TPA termination date.
# This is the dollar exposure calculation that gets the plan sponsor's attention.
df_claims = spark.table("pbm_sim.pharmacy_claims")

df_post_term_claims = (
    df_pbm_active_tpa_termed
    .select("member_id", "tpa_term_date")
    .join(df_claims, on="member_id")
    .filter(F.col("service_date") > F.col("tpa_term_date"))
)

post_term_count = df_post_term_claims.count()
post_term_dollars = df_post_term_claims.agg(F.sum("plan_paid")).collect()[0][0]

if post_term_count > 0:
    print(f"Claims filled after TPA term date: {post_term_count}")
    print(f"Total plan_paid on those claims:   ${post_term_dollars:,.2f}")
    print(f"\nThese represent potential overpayments that need review.")

    display(
        df_post_term_claims
        .select(
            "member_id", "claim_id", "service_date", "tpa_term_date",
            "drug_name", "drug_cost", "plan_paid"
        )
        .orderBy("member_id", "service_date")
    )

    log_entry(NB, "post_term_claims", "WARNING",
              row_count=post_term_count,
              detail=f"{post_term_count} claims totaling ${post_term_dollars:,.2f} "
                     f"filled after TPA termination date")
else:
    print("No claims found after TPA termination dates.")
    log_entry(NB, "post_term_claims", "SUCCESS",
              detail="No post-termination claims found")

No claims found after TPA termination dates.
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:33.631994+00:00", "notebook": "NB_02", "section": "post_term_claims", "status": "SUCCESS", "row_count": 0, "detail": "No post-termination claims found", "duration_sec": 0.0}


## 3. Effective Date Mismatches

If the PBM and TPA disagree on when a member's coverage started, it affects which
claims are considered "in-plan" and which benefit year the accumulators
should be attributed to.

Small differences of 1-3 days usually point to date parsing issues
between systems (e.g., one system interprets the 834's coverage start
as the first day of the month, the other uses the exact date in the file).
Larger differences suggest a more fundamental data problem.

In [0]:
df_eff_date_mismatch = df_joined.filter(
    F.col("pbm_effective_date") != F.col("tpa_effective_date")
)

eff_mismatch_count = df_eff_date_mismatch.count()
print(f"Members with effective date mismatch: {eff_mismatch_count}")

if eff_mismatch_count > 0:
    df_eff_detail = (
        df_eff_date_mismatch
        .withColumn(
            "day_difference",
            F.datediff(F.col("pbm_effective_date"), F.col("tpa_effective_date"))
        )
        .select(
            "member_id", "plan_id",
            "pbm_effective_date", "tpa_effective_date", "day_difference"
        )
        .orderBy(F.abs(F.col("day_difference")).desc())
    )
    display(df_eff_detail)

    # Distribution of how far off the dates are.
    # If most mismatches are 1-day differences, that's probably a systemic
    # parsing issue. If they're all over the map, it's something else.
    print("\nDistribution of day differences:")
    display(
        df_eff_detail
        .groupBy("day_difference")
        .agg(F.count("*").alias("member_count"))
        .orderBy("day_difference")
    )

log_entry(NB, "effective_date_mismatch",
          "WARNING" if eff_mismatch_count > 0 else "SUCCESS",
          row_count=eff_mismatch_count,
          detail=f"{eff_mismatch_count} members with effective date disagreement")

Members with effective date mismatch: 200


member_id,plan_id,pbm_effective_date,tpa_effective_date,day_difference
MBR1015747,PLN008,2026-01-01,2026-01-04,-3
MBR1048364,PLN039,2026-01-01,2026-01-04,-3
MBR1039195,PLN028,2026-01-01,2026-01-04,-3
MBR1006487,PLN003,2026-01-01,2026-01-04,-3
MBR1009498,PLN005,2026-07-01,2026-07-04,-3
MBR1016544,PLN009,2026-01-01,2026-01-04,-3
MBR1044234,PLN034,2026-01-01,2026-01-04,-3
MBR1001103,PLN001,2026-01-01,2026-01-04,-3
MBR1020684,PLN011,2026-01-01,2026-01-04,-3
MBR1033938,PLN022,2026-01-01,2026-01-04,-3



Distribution of day differences:


day_difference,member_count
-3,40
-2,35
-1,43
1,38
2,44


[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:38.771989+00:00", "notebook": "NB_02", "section": "effective_date_mismatch", "status": "WARNING", "row_count": 200, "detail": "200 members with effective date disagreement", "duration_sec": 0.0}


## 4. Plan ID Mismatches

If a member is assigned to different plan_ids across systems, the accumulator limits being applied are for the wrong
benefit design. A member on a $2,000 deductible plan in the TPA but
mapped to a $3,500 deductible plan in the PBM will hit their actual
limit $1,500 before the PBM realizes it.

In [0]:
df_plan_mismatch = (
    df_pbm.alias("p")
    .join(df_tpa.alias("t"), on="member_id", how="inner")
    .filter(F.col("p.plan_id") != F.col("t.plan_id"))
    .select(
        F.col("p.member_id"),
        F.col("p.plan_id").alias("pbm_plan_id"),
        F.col("t.plan_id").alias("tpa_plan_id"),
        F.col("p.first_name"),
        F.col("p.last_name"),
    )
)

plan_mismatch_count = df_plan_mismatch.count()
print(f"Members with plan_id mismatch: {plan_mismatch_count}")

if plan_mismatch_count > 0:
    display(df_plan_mismatch)
else:
    print(">> All matched members are on the same plan_id in both systems.")

log_entry(NB, "plan_id_mismatch",
          "WARNING" if plan_mismatch_count > 0 else "SUCCESS",
          row_count=plan_mismatch_count,
          detail=f"{plan_mismatch_count} members with plan_id disagreement")

Members with plan_id mismatch: 0
>> All matched members are on the same plan_id in both systems.
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:41.369135+00:00", "notebook": "NB_02", "section": "plan_id_mismatch", "status": "SUCCESS", "row_count": 0, "detail": "0 members with plan_id disagreement", "duration_sec": 0.0}


## Reconciliation Summary

Everything above rolls up into a single view. In production, the CRITICAL
findings get escalated immediately - especially the termination mismatches,
because every day a stale termination sits unfixed is another day the member
can fill prescriptions the plan sponsor will have to recover later.

In [0]:
results = [
    ("Members in PBM only (missing from TPA)", in_pbm_only_count,
     "HIGH" if in_pbm_only_count > 0 else "OK",
     "No medical accumulator data for these members"),

    ("Members in TPA only (missing from PBM)", in_tpa_only_count,
     "MEDIUM" if in_tpa_only_count > 0 else "OK",
     "TPA reporting on unknown members"),

    ("Active in PBM but terminated in TPA", pbm_active_tpa_termed_count,
     "CRITICAL" if pbm_active_tpa_termed_count > 0 else "OK",
     "May be filling claims on expired coverage"),

    ("Terminated in PBM but active in TPA", pbm_termed_tpa_active_count,
     "MEDIUM" if pbm_termed_tpa_active_count > 0 else "OK",
     "Member may be denied pharmacy coverage incorrectly"),

    ("Both termed, different term dates", term_date_mismatch_count,
     "LOW" if term_date_mismatch_count > 0 else "OK",
     "Timing inconsistency in termination processing"),

    ("Effective date mismatches", eff_mismatch_count,
     "MEDIUM" if eff_mismatch_count > 0 else "OK",
     "Date parsing difference between systems"),

    ("Plan ID mismatches", plan_mismatch_count,
     "CRITICAL" if plan_mismatch_count > 0 else "OK",
     "Wrong benefit design applied to member"),
]

summary_data = [(check, count, severity, impact) for check, count, severity, impact in results]
df_summary = spark.createDataFrame(summary_data, ["check", "discrepancies_found", "severity", "operational_impact"])

display(df_summary)

total_discrepancies = sum(count for _, count, _, _ in results)
print(f"\nTotal discrepancies across all checks: {total_discrepancies}")

log_entry(NB, "reconciliation_summary", "SUCCESS",
          row_count=total_discrepancies,
          detail=f"Total discrepancies: {total_discrepancies} across 7 checks")

check,discrepancies_found,severity,operational_impact
Members in PBM only (missing from TPA),125,HIGH,No medical accumulator data for these members
Members in TPA only (missing from PBM),0,OK,TPA reporting on unknown members
Active in PBM but terminated in TPA,500,CRITICAL,May be filling claims on expired coverage
Terminated in PBM but active in TPA,0,OK,Member may be denied pharmacy coverage incorrectly
"Both termed, different term dates",0,OK,Timing inconsistency in termination processing
Effective date mismatches,200,MEDIUM,Date parsing difference between systems
Plan ID mismatches,0,OK,Wrong benefit design applied to member



Total discrepancies across all checks: 825
[PIPELINE_LOG] {"run_id": "RUN_20261017_191803", "timestamp": "2026-10-17T19:18:43.952965+00:00", "notebook": "NB_02", "section": "reconciliation_summary", "status": "SUCCESS", "row_count": 825, "detail": "Total discrepancies: 825 across 7 checks", "duration_sec": 0.0}


## Investigation Notes

The standard workflow for each finding category:

**Members missing from TPA** need to be escalated to the TPA with a
request to confirm enrollment status. If they're genuinely enrolled, the
TPA needs to add them to the next feed. If they were terminated on the
TPA side without the PBM being notified, the PBM needs an updated 834.

**Termination mismatches** are the most urgent. Pull the original 834 to
confirm which date is correct. If the TPA's termination date is right,
the PBM needs to process the term immediately and any claims filled after
that date need to be flagged for reversal review.

**Effective date differences** of 1-3 days are typically systemic (date
parsing logic) rather than member-specific. If the pattern is consistent
across a plan, escalate to the enrollment ops team to align parsing rules.
If it's sporadic, check the individual 834 segments for those members.

In [0]:
log_summary(NB)


  PIPELINE LOG SUMMARY: NB_02 (run: RUN_20261017_191803)
  [+] load_source_tables: SUCCESS
  [!] members_pbm_only: WARNING (125 rows)
      125 members in PBM but absent from TPA feed
  [+] members_tpa_only: SUCCESS
  [!] term_pbm_active_tpa_termed: WARNING (500 rows)
      500 members active in PBM but termed in TPA
  [+] term_pbm_termed_tpa_active: SUCCESS
  [+] term_date_mismatch: SUCCESS
  [+] post_term_claims: SUCCESS
  [!] effective_date_mismatch: WARNING (200 rows)
      200 members with effective date disagreement
  [+] plan_id_mismatch: SUCCESS
  [+] reconciliation_summary: SUCCESS (825 rows)

